# Modèle final : ensemble LightGBM + XGBoost + Timestral (1/3 chacun)

Même méthode que le rendu (`model/modele_ensemble.py`) : on rejoue le challenge à un vendredi sur 4 (1 jour de trou, puis 120 heures notées), puis sur le test final, contre le naïf `lag_7j`.

In [ ]:
import sys; sys.path += ["..", "../model"]  # pour importer dataset_create et les modèles depuis notebooks/
import pandas as pd
from commun import jeu_final, jeu_val, rmse
from dataset_create import ORIGINES, charger
from modele_ensemble import MODELES

df = charger(imputer=True)
CIBLE = "debit"  # ou "occupation"

In [ ]:
def comparer(full, t0, test):
    P = {n: m(full, t0, CIBLE).clip(0) for n, m in MODELES.items()}
    P["ensemble"] = pd.concat(P.values(), axis=1).mean(axis=1)  # = modele_ensemble.predire
    P["naif_lag7j"] = full.lag_7j[full.datetime >= t0]
    return pd.DataFrame({n: rmse(full, test, p) for n, p in P.items()})

table = lambda t: pd.concat([t, t.mean().to_frame("moyenne").T]).round(1)

In [ ]:
# Validation : 1 vendredi sur 4, RMSE par arc moyennée sur les semaines
val = [comparer(*jeu_val(df, CIBLE, t0)) for t0 in ORIGINES[::4]]
table(pd.concat(val).groupby(level=0).mean())

In [ ]:
# Test final : de la fin des données (01/11/2026) à la dernière vraie valeur téléchargée
table(comparer(*jeu_final(df, CIBLE)))